# Forces

GNNs can predict the potential energy surface of molecules or crystals as a function of composition and geometry using models like SchNet or PAiNN. If the GNN is differentiable (using smooth activations like `swish` or `shifted_softplus`), then automatic differentiation in PyTorch can derive forces as the negative gradient of energy:

$$\vec{F} = -\nabla\, E$$

If reference gradients are available from ab-initio calculations (e.g. DFT), models can be trained on both energy and force targets by combining their losses with weights $\alpha$ and $\beta$:

$$\mathcal{L} = \alpha\; \mathcal{L}(E, \hat{E}) + \beta\; \mathcal{L}(\vec{F}, \hat{\vec{F}})$$

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader

## EnergyForceModel

The `kgcnn_torch.models.force.EnergyForceModel` wraps any energy-predicting GNN to also predict forces via `torch.autograd.grad`. The key requirement is that `data.pos` must have `requires_grad=True`.

In [ ]:
from kgcnn_torch.models.schnet import SchNetModel
from kgcnn_torch.models.force import EnergyForceModel

# 1. Create an energy-predicting SchNet model
energy_model = SchNetModel(
    node_dim=64,
    depth=4,
    units=128,
    gauss_bins=25,
    gauss_distance=5.0,
    gauss_sigma=0.4,
    interaction_activation="shifted_softplus",
    interaction_pooling="sum",
    node_pooling="sum",
    last_mlp_units=[128, 64],
    num_targets=1,
    make_distance=True,
    expand_distance=True,
    use_output_mlp=False,  # Energy output from last MLP
)

# 2. Wrap with EnergyForceModel
model = EnergyForceModel(
    energy_model=energy_model,
    coordinate_input="pos",         # Name of coordinate attribute on data
    output_as_dict=True,            # Return {'energy': ..., 'force': ...}
    output_squeeze_states=True,
    is_physical_force=True,         # F = -grad(E)
)

print("EnergyForceModel config:", model.get_config())

## Forward Pass: Computing Energy and Forces

The forward pass computes energy from the inner model, then uses `torch.autograd.grad` to compute forces.

In [ ]:
# Create example molecular data
# Ethanol-like: C2OH6 (9 atoms)
data1 = Data(
    z=torch.tensor([6, 6, 8, 1, 1, 1, 1, 1, 1]),
    pos=torch.tensor([
        [0.000, 0.591, 0.000],
        [-1.254, -0.255, -0.030],
        [1.088, -0.308, 0.048],
        [0.063, 1.284, -0.843],
        [0.006, 1.230, 0.885],
        [-2.218, 0.190, -0.058],
        [-0.911, -1.054, -0.782],
        [-1.192, -0.742, 0.922],
        [1.849, -0.029, -0.526],
    ], dtype=torch.float32, requires_grad=True),  # requires_grad=True!
)

# Build edges: fully connect nearby atoms
n = data1.z.size(0)
src, tgt = [], []
for i in range(n):
    for j in range(n):
        if i != j:
            src.append(i)
            tgt.append(j)
data1.edge_index = torch.tensor([src, tgt])

batch = Batch.from_data_list([data1])
# Ensure pos requires gradient after batching
batch.pos.requires_grad_(True)

# Forward pass
result = model(batch)

print("Energy:", result['energy'].item())
print("Forces shape:", result['force'].shape)  # (N, 3)
print("Forces (first 3 atoms):")
print(result['force'][:3].detach().numpy())

## Using PAiNN as Energy Model

PAiNN is an equivariant model that naturally works well for force prediction since it explicitly uses directional information.

In [ ]:
from kgcnn_torch.models.painn import PAiNNModel

# PAiNN energy model
painn_energy = PAiNNModel(
    node_dim=128,
    depth=3,
    units=128,
    num_radial=20,
    cutoff=5.0,
    conv_activation="swish",
    update_activation="swish",
    node_pooling="sum",
    output_units=[128],
    num_targets=1,
)

# Wrap with force model
painn_force = EnergyForceModel(
    energy_model=painn_energy,
    coordinate_input="pos",
    output_as_dict=True,
    is_physical_force=True,
)

batch_painn = Batch.from_data_list([data1])
batch_painn.pos.requires_grad_(True)

result_painn = painn_force(batch_painn)
print("PAiNN Energy:", result_painn['energy'].item())
print("PAiNN Forces shape:", result_painn['force'].shape)

## Creating a Force Dataset

For training force models, we need a dataset with energy and force labels. Here we show how to construct such a dataset in PyG format.

In [ ]:
def create_force_data(z, pos, energy, forces, cutoff=5.0):
    """Create a PyG Data object for force model training.
    
    Args:
        z: Atomic numbers, shape (N,).
        pos: Atom positions, shape (N, 3).
        energy: Scalar energy value.
        forces: Force vectors, shape (N, 3).
        cutoff: Distance cutoff for edges.
    
    Returns:
        PyG Data object with energy (y) and force labels.
    """
    z_t = torch.tensor(z, dtype=torch.long)
    pos_t = torch.tensor(pos, dtype=torch.float32)
    
    # Build edges within cutoff
    n = len(z)
    src, tgt = [], []
    for i in range(n):
        for j in range(n):
            if i != j:
                dist = np.linalg.norm(pos[i] - pos[j])
                if dist < cutoff:
                    src.append(i)
                    tgt.append(j)
    edge_index = torch.tensor([src, tgt], dtype=torch.long)
    
    data = Data(
        z=z_t,
        pos=pos_t,
        edge_index=edge_index,
        y=torch.tensor([energy], dtype=torch.float32),
        force=torch.tensor(forces, dtype=torch.float32),
    )
    return data


# Create synthetic force training data
# In real usage, this comes from DFT calculations
np.random.seed(42)
z_atoms = [6, 6, 8, 1, 1, 1, 1, 1, 1]  # Ethanol-like
force_dataset = []

for i in range(50):
    # Perturb positions (simulating MD snapshots)
    base_pos = np.array([
        [0.0, 0.59, 0.0], [-1.25, -0.26, -0.03], [1.09, -0.31, 0.05],
        [0.06, 1.28, -0.84], [0.01, 1.23, 0.89], [-2.22, 0.19, -0.06],
        [-0.91, -1.05, -0.78], [-1.19, -0.74, 0.92], [1.85, -0.03, -0.53],
    ])
    pos = base_pos + np.random.randn(*base_pos.shape) * 0.1
    
    # Synthetic energy and forces (in real usage from DFT)
    energy = -154.0 + np.random.randn() * 0.1
    forces = np.random.randn(9, 3) * 0.5
    
    force_dataset.append(create_force_data(z_atoms, pos, energy, forces))

print(f"Force dataset: {len(force_dataset)} configurations")
print(f"  Example: z={force_dataset[0].z.tolist()}, E={force_dataset[0].y.item():.2f}")
print(f"  Force shape: {force_dataset[0].force.shape}")

## Training the Force Model

Training requires a combined loss function that handles both energy and force predictions.

In [ ]:
def energy_force_loss(pred_dict, batch, energy_weight=0.02, force_weight=0.98):
    """Combined energy + force loss function.
    
    Args:
        pred_dict: Dict with 'energy' (B, 1) and 'force' (N, 3) predictions.
        batch: PyG Batch object with y (energy labels) and force (force labels).
        energy_weight: Weight for energy loss.
        force_weight: Weight for force loss.
    
    Returns:
        Weighted combined loss.
    """
    # Energy loss: MAE over batch
    pred_energy = pred_dict['energy']
    true_energy = batch.y
    if true_energy.dim() == 1:
        true_energy = true_energy.unsqueeze(-1)
    loss_energy = torch.nn.functional.l1_loss(pred_energy, true_energy)
    
    # Force loss: MAE over all atoms and components
    pred_force = pred_dict['force']
    true_force = batch.force
    loss_force = torch.nn.functional.l1_loss(pred_force, true_force)
    
    return energy_weight * loss_energy + force_weight * loss_force

In [ ]:
# Split data
train_data = force_dataset[:40]
val_data = force_dataset[40:]

train_loader = DataLoader(train_data, batch_size=8, shuffle=True)
val_loader = DataLoader(val_data, batch_size=8, shuffle=False)

# Create force model
schnet_energy = SchNetModel(
    node_dim=32, depth=3, units=64,
    gauss_bins=20, gauss_distance=5.0,
    num_targets=1, use_output_mlp=False,
)

force_model = EnergyForceModel(
    energy_model=schnet_energy,
    coordinate_input="pos",
    output_as_dict=True,
    is_physical_force=True,
)

optimizer = torch.optim.Adam(force_model.parameters(), lr=1e-3)

print("Starting force model training...")

In [ ]:
# Manual training loop for force models
# (The trainer.fit function also supports dict-output models)

n_epochs = 30
for epoch in range(n_epochs):
    force_model.train()
    train_loss = 0.0
    n_batch = 0
    
    for batch in train_loader:
        # Ensure positions require gradient for autograd force computation
        batch.pos.requires_grad_(True)
        
        optimizer.zero_grad()
        pred = force_model(batch)
        loss = energy_force_loss(pred, batch)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        n_batch += 1
    
    avg_loss = train_loss / n_batch
    
    # Validation
    if (epoch + 1) % 10 == 0:
        force_model.eval()
        val_loss = 0.0
        n_val = 0
        for batch in val_loader:
            batch.pos.requires_grad_(True)
            with torch.set_grad_enabled(True):  # Need grad for forces
                pred = force_model(batch)
            loss = energy_force_loss(pred, batch)
            val_loss += loss.item()
            n_val += 1
        
        print(f"Epoch {epoch+1}/{n_epochs} - "
              f"train_loss: {avg_loss:.4f} - val_loss: {val_loss/n_val:.4f}")

## Using trainer.fit() with Force Models

The `kgcnn_torch.training.trainer.fit()` function supports force models that return dict predictions. The loss function receives `(pred_dict, batch)` instead of `(pred, target)`.

In [ ]:
from kgcnn_torch.training.trainer import fit

# Recreate model
schnet_e = SchNetModel(
    node_dim=32, depth=3, units=64,
    gauss_bins=20, gauss_distance=5.0,
    num_targets=1, use_output_mlp=False,
)
fm = EnergyForceModel(
    energy_model=schnet_e,
    coordinate_input="pos",
    output_as_dict=True,
    is_physical_force=True,
)

# Ensure positions have requires_grad in the dataset
for d in train_data:
    d.pos.requires_grad_(True)
for d in val_data:
    d.pos.requires_grad_(True)

history = fit(
    model=fm,
    train_loader=DataLoader(train_data, batch_size=8, shuffle=True),
    val_loader=DataLoader(val_data, batch_size=8, shuffle=False),
    optimizer=torch.optim.Adam(fm.parameters(), lr=1e-3),
    loss_fn=energy_force_loss,  # Accepts (pred_dict, batch)
    epochs=30,
    verbose=1,
)

print(f"\nFinal train loss: {history['train_loss'][-1]:.4f}")
print(f"Final val loss: {history['val_loss'][-1]:.4f}")

## Evaluating Force Predictions

After training, we can evaluate the model by comparing predicted forces against reference values.

In [ ]:
# Evaluate on validation set
fm.eval()
all_pred_forces = []
all_true_forces = []
all_pred_energy = []
all_true_energy = []

for batch in DataLoader(val_data, batch_size=len(val_data)):
    batch.pos.requires_grad_(True)
    with torch.set_grad_enabled(True):
        pred = fm(batch)
    
    all_pred_forces.append(pred['force'].detach().numpy())
    all_true_forces.append(batch.force.numpy())
    all_pred_energy.append(pred['energy'].detach().numpy())
    all_true_energy.append(batch.y.numpy())

pred_f = np.concatenate(all_pred_forces)
true_f = np.concatenate(all_true_forces)
pred_e = np.concatenate(all_pred_energy)
true_e = np.concatenate(all_true_energy)

# MAE for energy and forces
mae_energy = np.mean(np.abs(pred_e - true_e))
mae_forces = np.mean(np.abs(pred_f - true_f))

print(f"Energy MAE: {mae_energy:.4f}")
print(f"Force MAE: {mae_forces:.4f}")
print(f"Force components shape: {pred_f.shape}")

## Saving and Loading Force Models

Force models can be saved and loaded using standard PyTorch methods.

In [ ]:
# Save model
# torch.save(fm.state_dict(), "force_model.pt")

# Load model
# fm.load_state_dict(torch.load("force_model.pt", weights_only=True))

print("Model can be saved/loaded with torch.save() and torch.load().")
print("Total parameters:", sum(p.numel() for p in fm.parameters()))
print("Trainable parameters:", sum(p.numel() for p in fm.parameters() if p.requires_grad))

## Notes on Force Models

**Important considerations:**

1. **requires_grad**: The coordinate tensor (`data.pos`) must have `requires_grad=True` for autograd force computation. The `EnergyForceModel` sets this automatically if not already set.

2. **Smooth activations**: Use smooth activations like `swish` or `shifted_softplus` instead of `relu` for energy models. Non-smooth activations lead to discontinuous force surfaces.

3. **Training mode**: During training, `create_graph=True` is used in `torch.autograd.grad` to allow backpropagation through the force computation (needed for force loss gradients).

4. **Evaluation**: Force models need gradient computation even during evaluation, so `torch.no_grad()` cannot be used. The `trainer.eval_epoch` handles this automatically.

5. **Loss weights**: The energy/force loss weights ($\alpha$ and $\beta$) significantly affect training dynamics. Typical values are `energy_weight=0.01-0.05` and `force_weight=0.95-0.99`.

6. **Equivariant models**: Models like PAiNN that maintain equivariant features often perform better for force prediction than invariant models like SchNet.

> **NOTE**: You can find this page as a Jupyter notebook in the `docs/source` directory of the kgcnn-torch repository.